# MLMC Stress Test Parameter Exploration

**Purpose**: Find parameters that create a complex enough volatility surface to require meaningful contributions from finer MLMC levels.

**Problem**: With the paper's Eq. 56 parameters (d=3, σ~0.15, T=0.5), Level 0 captures ~99.9% of the surface, making α and β fits meaningless.

**Goal**: Find parameters where Level 0 contributes ~60%, Level 1 ~25%, Level 2 ~10%, Level 3 ~5%.

---

**Author**: Wadoud (KAUST Internship)  
**Date**: December 2024

In [ ]:
# =============================================================================
# CONFIGURATION
# =============================================================================

# Toggle to save plots externally (False = inline only)
SAVE_PLOTS = False

# Output directory for saved plots (only used if SAVE_PLOTS=True)
SAVE_DIR = "results/stress_test_exploration"

In [ ]:
# =============================================================================
# IMPORTS AND PATH SETUP
# =============================================================================

import sys
from pathlib import Path

# Ensure we're importing from the right location
# This notebook runs from the Comparison_Between_Methods folder
_notebook_dir = Path.cwd()
if str(_notebook_dir) not in sys.path:
    sys.path.insert(0, str(_notebook_dir))

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import warnings
warnings.filterwarnings('ignore')

# Display plots inline
%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 11

# Import from our modules
from config import (
    ProblemParameters, 
    DEFAULT_PARAMS,
    create_nd_params,
    create_2d_params,
    create_5d_params,
    create_10d_params
)
from methods.mlmc_ot_estimator import (
    estimate_domain,
    make_c_ot,
    make_c,
    tot_degree_poly,
    make_b_squared
)
from methods.common import compute_mlmc_level_diagnostics

print("Imports successful!")
print(f"Working directory: {_notebook_dir}")

In [ ]:
# =============================================================================
# HELPER FUNCTIONS
# =============================================================================

def save_figure(fig, name):
    """Save figure if SAVE_PLOTS is True."""
    if SAVE_PLOTS:
        save_path = Path(SAVE_DIR)
        save_path.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_path / f"{name}.png", dpi=150, bbox_inches='tight')
        print(f"Saved: {save_path / name}.png")


def run_mlmc_diagnostics(params, verbose=True):
    """
    Run MLMC with diagnostics and return level statistics.
    
    Returns
    -------
    dict with keys:
        - level_stats: per-level LevelStats objects
        - diagnostics: computed diagnostics (alpha, beta, etc.)
        - corrections: list of mean corrections per level
        - level_fractions: fraction of total correction per level
        - coefficients: final coefficient vector
    """
    np.random.seed(params.random_seed)
    
    # Estimate domain
    s_min, s_max = estimate_domain(
        x0=params.x0,
        T=params.T,
        h0=params.h0,
        r=params.r,
        cov_mat=params.corr_matrix,
        vol=params.sigma,
        max_deg=params.max_degree,
        P1=params.P1,
        M_pilot=10000
    )
    
    # Add padding
    pad = (s_max - s_min) * 0.05
    s_min -= pad
    s_max += pad
    
    if verbose:
        print(f"Domain: [{s_min:.1f}, {s_max:.1f}]")
    
    # Run MLMC with statistics
    c, level_stats = make_c_ot(
        x0=params.x0,
        T=params.T,
        h0=params.h0,
        r=params.r,
        cov_mat=params.corr_matrix,
        vol=params.sigma,
        max_deg=params.max_degree,
        P1=params.P1,
        s_min=s_min,
        s_max=s_max,
        C=80,
        batch_size=50,
        verbose=verbose,
        return_stats=True
    )
    
    # Compute diagnostics
    diagnostics = compute_mlmc_level_diagnostics(level_stats)
    
    # Extract corrections and compute fractions
    n_levels = params.max_degree + 1
    corrections = [level_stats[l].mean_correction for l in range(n_levels)]
    total_abs = sum(abs(c) for c in corrections)
    
    if total_abs > 1e-10:
        level_fractions = [abs(c) / total_abs for c in corrections]
    else:
        level_fractions = [1.0 / n_levels] * n_levels
    
    return {
        'level_stats': level_stats,
        'diagnostics': diagnostics,
        'corrections': corrections,
        'level_fractions': level_fractions,
        'coefficients': c,
        's_min': s_min,
        's_max': s_max
    }


def plot_level_diagnostics(results, params, title_suffix=""):
    """
    Create a 2x2 diagnostic plot similar to exp2_mlmc_diagnostics.
    """
    level_stats = results['level_stats']
    diagnostics = results['diagnostics']
    
    n_levels = params.max_degree + 1
    levels = list(range(n_levels))
    
    # Extract data
    corrections = results['corrections']
    variances = [level_stats[l].variance for l in levels]
    correlations = [level_stats[l].correlation for l in levels]
    
    alpha = diagnostics['alpha']
    beta = diagnostics['beta']
    quality = diagnostics['convergence_quality']
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle(f"MLMC Convergence Diagnostics (Quality: {quality.upper()}){title_suffix}", 
                 fontsize=14, fontweight='bold', color='darkorange' if quality != 'good' else 'green')
    
    # Panel 1: Weak convergence (log₂|m_ℓ|)
    ax1 = axes[0, 0]
    log2_m = []
    lvls_m = []
    for l in levels:
        m_l = abs(corrections[l])
        if m_l > 1e-15:
            log2_m.append(np.log2(m_l))
            lvls_m.append(l)
    
    if len(lvls_m) > 0:
        ax1.plot(lvls_m, log2_m, 'bo-', markersize=8, linewidth=2, label='Data')
        
        # Fitted line for levels >= 1
        if not np.isnan(alpha) and len([l for l in lvls_m if l >= 1]) >= 2:
            l_fit = np.array([l for l in lvls_m if l >= 1])
            log2_m_fit = np.array([log2_m[i] for i, l in enumerate(lvls_m) if l >= 1])
            if len(l_fit) >= 2:
                l_line = np.linspace(min(l_fit), max(l_fit), 50)
                intercept = np.mean(log2_m_fit) + alpha * np.mean(l_fit)
                fit_line = intercept - alpha * l_line
                ax1.plot(l_line, fit_line, 'r--', linewidth=1.5,
                         label=rf'Fit: $\alpha = {alpha:.2f}$')
    
    ax1.set_xlabel(r'Level $\ell$')
    ax1.set_ylabel(r'$\log_2 |m_\ell|$')
    ax1.set_title(r'Weak Convergence: $|m_\ell| = \|b^2_\ell - b^2_{\ell-1}\|$')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    ax1.set_xticks(levels)
    
    # Panel 2: Variance decay (log₂V_ℓ)
    ax2 = axes[0, 1]
    higher_levels = [l for l in levels if l >= 1]
    log2_v = []
    lvls_v = []
    for l in higher_levels:
        v_l = variances[l]
        if v_l > 1e-15:
            log2_v.append(np.log2(v_l))
            lvls_v.append(l)
    
    if len(lvls_v) > 0:
        ax2.plot(lvls_v, log2_v, 'go-', markersize=8, linewidth=2, label='Data')
        
        if not np.isnan(beta) and len(lvls_v) >= 2:
            l_line = np.linspace(min(lvls_v), max(lvls_v), 50)
            intercept = np.mean(log2_v) + beta * np.mean(lvls_v)
            fit_line = intercept - beta * l_line
            ax2.plot(l_line, fit_line, 'r--', linewidth=1.5,
                     label=rf'Fit: $\beta = {beta:.2f}$')
    
    ax2.set_xlabel(r'Level $\ell$')
    ax2.set_ylabel(r'$\log_2 V_\ell$')
    ax2.set_title(r'Variance Decay: $V_\ell = \mathrm{Var}[P_\ell - P_{\ell-1}]$')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    if len(lvls_v) > 0:
        ax2.set_xticks(lvls_v)
    
    # Panel 3: Level corrections bar chart
    ax3 = axes[1, 0]
    colors = ['C0' if c >= 0 else 'C3' for c in corrections]
    bars = ax3.bar(levels, corrections, color=colors, edgecolor='black', alpha=0.7)
    ax3.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
    ax3.set_xlabel(r'Level $\ell$')
    ax3.set_ylabel(r'Mean correction $m_\ell$')
    ax3.set_title('Level Corrections (Telescoping Sum)')
    ax3.grid(True, alpha=0.3, axis='y')
    ax3.set_xticks(levels)
    
    # Add percentage labels
    fractions = results['level_fractions']
    for i, (bar, frac) in enumerate(zip(bars, fractions)):
        height = bar.get_height()
        ax3.annotate(f'{frac*100:.1f}%',
                    xy=(bar.get_x() + bar.get_width()/2, height),
                    xytext=(0, 3 if height >= 0 else -12),
                    textcoords="offset points",
                    ha='center', va='bottom' if height >= 0 else 'top',
                    fontsize=10, fontweight='bold')
    
    # Panel 4: Correlation
    ax4 = axes[1, 1]
    corr_levels = [l for l in levels if l >= 1]
    corr_values = [correlations[l] for l in corr_levels]
    ax4.plot(corr_levels, corr_values, 'mo-', markersize=10, linewidth=2)
    ax4.axhline(y=0.95, color='red', linestyle='--', linewidth=1.5, 
                label=r'Threshold $\rho = 0.95$')
    ax4.set_xlabel(r'Level $\ell$')
    ax4.set_ylabel(r'Correlation $\rho_\ell$')
    ax4.set_title(r'Coupling Correlation: $\rho_\ell = \mathrm{Corr}(P_\ell, P_{\ell-1})$')
    ax4.legend(loc='lower right')
    ax4.grid(True, alpha=0.3)
    ax4.set_ylim(0.7, 1.02)
    if len(corr_levels) > 0:
        ax4.set_xticks(corr_levels)
    
    plt.tight_layout()
    return fig


def create_custom_params(
    d: int = 10,
    base_sigma: float = 0.35,
    rho: float = 0.85,
    T: float = 1.5,
    max_degree: int = 4,
    h0: float = None,
    seed: int = 42
) -> ProblemParameters:
    """
    Create custom parameters for stress testing.
    
    If h0 is None, it's computed to satisfy stability constraint:
        h0 ≤ T / (2*max_degree + 5)
    """
    np.random.seed(seed)
    
    # Varying volatilities
    sigma_multipliers = 0.7 + 0.6 * np.random.rand(d)
    sigma = base_sigma * sigma_multipliers
    
    # Toeplitz correlation matrix
    corr_matrix = np.zeros((d, d))
    for i in range(d):
        for j in range(d):
            corr_matrix[i, j] = rho ** abs(i - j)
    
    # Compute safe h0 if not provided
    if h0 is None:
        h0_max = T / (2 * max_degree + 5)
        h0 = 0.8 * h0_max  # 20% safety margin
    
    return ProblemParameters(
        r=0.05,
        sigma=sigma,
        corr_matrix=corr_matrix,
        P1=np.ones(d),
        x0=100.0 * np.ones(d),
        T=T,
        K=100.0 * d,
        max_degree=max_degree,
        h0=h0,
        random_seed=seed
    )

print("Helper functions defined!")

---

## 1. Baseline: Paper's Eq. 56 Parameters

First, let's confirm the problem with the default parameters.

In [ ]:
print("=" * 70)
print("BASELINE: Paper's Equation 56 Parameters (d=3)")
print("=" * 70)
print()
print(DEFAULT_PARAMS)
print()

baseline_results = run_mlmc_diagnostics(DEFAULT_PARAMS, verbose=True)

print("\n" + "=" * 50)
print("LEVEL CONTRIBUTION SUMMARY")
print("=" * 50)
for l, frac in enumerate(baseline_results['level_fractions']):
    print(f"  Level {l}: {frac*100:6.2f}%")
print(f"\n  α = {baseline_results['diagnostics']['alpha']:.3f}")
print(f"  β = {baseline_results['diagnostics']['beta']:.3f}")
print(f"  Quality: {baseline_results['diagnostics']['convergence_quality']}")

In [ ]:
fig_baseline = plot_level_diagnostics(
    baseline_results, DEFAULT_PARAMS, 
    title_suffix="\n(Baseline: Paper Eq. 56, d=3)"
)
save_figure(fig_baseline, "baseline_eq56")
plt.show()

---

## 2. Parameter Sweep: Finding the Sweet Spot

Let's systematically vary parameters to find combinations that produce meaningful level contributions.

In [ ]:
# =============================================================================
# PARAMETER SWEEP
# =============================================================================

# Parameters to sweep
dimensions = [3, 5, 10, 15]
base_sigmas = [0.15, 0.25, 0.35, 0.45]
correlations = [0.3, 0.5, 0.7, 0.85]
maturities = [0.5, 1.0, 1.5, 2.0]

print("Starting parameter sweep...")
print(f"  Dimensions: {dimensions}")
print(f"  Base volatilities: {base_sigmas}")
print(f"  Correlations: {correlations}")
print(f"  Maturities: {maturities}")
print()

sweep_results = []

In [ ]:
# Sweep 1: Dimension effect (keeping other params moderate)
print("=" * 70)
print("SWEEP 1: Effect of Dimension d")
print("=" * 70)

dim_results = []
for d in dimensions:
    params = create_custom_params(
        d=d, base_sigma=0.25, rho=0.7, T=1.0, max_degree=3
    )
    
    print(f"\nd = {d}:")
    results = run_mlmc_diagnostics(params, verbose=False)
    
    dim_results.append({
        'd': d,
        'params': params,
        'results': results,
        'level_0_frac': results['level_fractions'][0],
        'alpha': results['diagnostics']['alpha'],
        'beta': results['diagnostics']['beta'],
        'quality': results['diagnostics']['convergence_quality']
    })
    
    print(f"  Level fractions: {[f'{f*100:.1f}%' for f in results['level_fractions']]}")
    print(f"  α = {results['diagnostics']['alpha']:.3f}, β = {results['diagnostics']['beta']:.3f}")
    print(f"  Quality: {results['diagnostics']['convergence_quality']}")

In [ ]:
# Visualise dimension sweep
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

d_vals = [r['d'] for r in dim_results]
l0_fracs = [r['level_0_frac'] * 100 for r in dim_results]
alphas = [r['alpha'] for r in dim_results]
betas = [r['beta'] for r in dim_results]

ax1 = axes[0]
ax1.bar(d_vals, l0_fracs, color='steelblue', edgecolor='black', alpha=0.7)
ax1.axhline(y=60, color='green', linestyle='--', linewidth=2, label='Target: 60%')
ax1.set_xlabel('Number of Assets (d)', fontsize=12)
ax1.set_ylabel('Level 0 Contribution (%)', fontsize=12)
ax1.set_title('Effect of Dimension on Level 0 Dominance', fontsize=13)
ax1.set_xticks(d_vals)
ax1.legend()
ax1.grid(True, alpha=0.3, axis='y')

ax2 = axes[1]
ax2.plot(d_vals, alphas, 'bo-', markersize=10, linewidth=2, label=r'$\alpha$')
ax2.plot(d_vals, betas, 'rs-', markersize=10, linewidth=2, label=r'$\beta$')
ax2.axhline(y=1.0, color='blue', linestyle=':', alpha=0.5, label=r'$\alpha$ target')
ax2.axhline(y=2.0, color='red', linestyle=':', alpha=0.5, label=r'$\beta$ target')
ax2.set_xlabel('Number of Assets (d)', fontsize=12)
ax2.set_ylabel('Convergence Rate', fontsize=12)
ax2.set_title('Convergence Rates vs Dimension', fontsize=13)
ax2.set_xticks(d_vals)
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
save_figure(fig, "sweep_dimension")
plt.show()

In [ ]:
# Sweep 2: Volatility effect
print("=" * 70)
print("SWEEP 2: Effect of Base Volatility σ")
print("=" * 70)

sigma_results = []
for sigma in base_sigmas:
    params = create_custom_params(
        d=10, base_sigma=sigma, rho=0.7, T=1.0, max_degree=3
    )
    
    print(f"\nσ_base = {sigma}:")
    results = run_mlmc_diagnostics(params, verbose=False)
    
    sigma_results.append({
        'sigma': sigma,
        'params': params,
        'results': results,
        'level_0_frac': results['level_fractions'][0],
        'alpha': results['diagnostics']['alpha'],
        'beta': results['diagnostics']['beta'],
        'quality': results['diagnostics']['convergence_quality']
    })
    
    print(f"  Level fractions: {[f'{f*100:.1f}%' for f in results['level_fractions']]}")
    print(f"  α = {results['diagnostics']['alpha']:.3f}, β = {results['diagnostics']['beta']:.3f}")

In [ ]:
# Visualise volatility sweep
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sigma_vals = [r['sigma'] for r in sigma_results]
l0_fracs = [r['level_0_frac'] * 100 for r in sigma_results]
alphas = [r['alpha'] for r in sigma_results]
betas = [r['beta'] for r in sigma_results]

ax1 = axes[0]
ax1.bar(range(len(sigma_vals)), l0_fracs, color='darkorange', edgecolor='black', alpha=0.7)
ax1.axhline(y=60, color='green', linestyle='--', linewidth=2, label='Target: 60%')
ax1.set_xlabel('Base Volatility σ', fontsize=12)
ax1.set_ylabel('Level 0 Contribution (%)', fontsize=12)
ax1.set_title('Effect of Volatility on Level 0 Dominance', fontsize=13)
ax1.set_xticks(range(len(sigma_vals)))
ax1.set_xticklabels([f'{s:.2f}' for s in sigma_vals])
ax1.legend()
ax1.grid(True, alpha=0.3, axis='y')

ax2 = axes[1]
ax2.plot(sigma_vals, alphas, 'bo-', markersize=10, linewidth=2, label=r'$\alpha$')
ax2.plot(sigma_vals, betas, 'rs-', markersize=10, linewidth=2, label=r'$\beta$')
ax2.axhline(y=1.0, color='blue', linestyle=':', alpha=0.5)
ax2.axhline(y=2.0, color='red', linestyle=':', alpha=0.5)
ax2.set_xlabel('Base Volatility σ', fontsize=12)
ax2.set_ylabel('Convergence Rate', fontsize=12)
ax2.set_title('Convergence Rates vs Volatility', fontsize=13)
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
save_figure(fig, "sweep_volatility")
plt.show()

In [ ]:
# Sweep 3: Correlation effect
print("=" * 70)
print("SWEEP 3: Effect of Correlation ρ")
print("=" * 70)

corr_results = []
for rho in correlations:
    params = create_custom_params(
        d=10, base_sigma=0.30, rho=rho, T=1.0, max_degree=3
    )
    
    print(f"\nρ = {rho}:")
    results = run_mlmc_diagnostics(params, verbose=False)
    
    corr_results.append({
        'rho': rho,
        'params': params,
        'results': results,
        'level_0_frac': results['level_fractions'][0],
        'alpha': results['diagnostics']['alpha'],
        'beta': results['diagnostics']['beta'],
        'quality': results['diagnostics']['convergence_quality']
    })
    
    print(f"  Level fractions: {[f'{f*100:.1f}%' for f in results['level_fractions']]}")
    print(f"  α = {results['diagnostics']['alpha']:.3f}, β = {results['diagnostics']['beta']:.3f}")

In [ ]:
# Visualise correlation sweep
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

rho_vals = [r['rho'] for r in corr_results]
l0_fracs = [r['level_0_frac'] * 100 for r in corr_results]
alphas = [r['alpha'] for r in corr_results]
betas = [r['beta'] for r in corr_results]

ax1 = axes[0]
ax1.bar(range(len(rho_vals)), l0_fracs, color='purple', edgecolor='black', alpha=0.7)
ax1.axhline(y=60, color='green', linestyle='--', linewidth=2, label='Target: 60%')
ax1.set_xlabel('Correlation ρ', fontsize=12)
ax1.set_ylabel('Level 0 Contribution (%)', fontsize=12)
ax1.set_title('Effect of Correlation on Level 0 Dominance', fontsize=13)
ax1.set_xticks(range(len(rho_vals)))
ax1.set_xticklabels([f'{r:.2f}' for r in rho_vals])
ax1.legend()
ax1.grid(True, alpha=0.3, axis='y')

ax2 = axes[1]
ax2.plot(rho_vals, alphas, 'bo-', markersize=10, linewidth=2, label=r'$\alpha$')
ax2.plot(rho_vals, betas, 'rs-', markersize=10, linewidth=2, label=r'$\beta$')
ax2.axhline(y=1.0, color='blue', linestyle=':', alpha=0.5)
ax2.axhline(y=2.0, color='red', linestyle=':', alpha=0.5)
ax2.set_xlabel('Correlation ρ', fontsize=12)
ax2.set_ylabel('Convergence Rate', fontsize=12)
ax2.set_title('Convergence Rates vs Correlation', fontsize=13)
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
save_figure(fig, "sweep_correlation")
plt.show()

In [ ]:
# Sweep 4: Maturity effect
print("=" * 70)
print("SWEEP 4: Effect of Maturity T")
print("=" * 70)

maturity_results = []
for T in maturities:
    params = create_custom_params(
        d=10, base_sigma=0.30, rho=0.7, T=T, max_degree=3
    )
    
    print(f"\nT = {T}:")
    results = run_mlmc_diagnostics(params, verbose=False)
    
    maturity_results.append({
        'T': T,
        'params': params,
        'results': results,
        'level_0_frac': results['level_fractions'][0],
        'alpha': results['diagnostics']['alpha'],
        'beta': results['diagnostics']['beta'],
        'quality': results['diagnostics']['convergence_quality']
    })
    
    print(f"  Level fractions: {[f'{f*100:.1f}%' for f in results['level_fractions']]}")
    print(f"  α = {results['diagnostics']['alpha']:.3f}, β = {results['diagnostics']['beta']:.3f}")

In [ ]:
# Visualise maturity sweep
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

T_vals = [r['T'] for r in maturity_results]
l0_fracs = [r['level_0_frac'] * 100 for r in maturity_results]
alphas = [r['alpha'] for r in maturity_results]
betas = [r['beta'] for r in maturity_results]

ax1 = axes[0]
ax1.bar(range(len(T_vals)), l0_fracs, color='teal', edgecolor='black', alpha=0.7)
ax1.axhline(y=60, color='green', linestyle='--', linewidth=2, label='Target: 60%')
ax1.set_xlabel('Maturity T (years)', fontsize=12)
ax1.set_ylabel('Level 0 Contribution (%)', fontsize=12)
ax1.set_title('Effect of Maturity on Level 0 Dominance', fontsize=13)
ax1.set_xticks(range(len(T_vals)))
ax1.set_xticklabels([f'{t:.1f}' for t in T_vals])
ax1.legend()
ax1.grid(True, alpha=0.3, axis='y')

ax2 = axes[1]
ax2.plot(T_vals, alphas, 'bo-', markersize=10, linewidth=2, label=r'$\alpha$')
ax2.plot(T_vals, betas, 'rs-', markersize=10, linewidth=2, label=r'$\beta$')
ax2.axhline(y=1.0, color='blue', linestyle=':', alpha=0.5)
ax2.axhline(y=2.0, color='red', linestyle=':', alpha=0.5)
ax2.set_xlabel('Maturity T (years)', fontsize=12)
ax2.set_ylabel('Convergence Rate', fontsize=12)
ax2.set_title('Convergence Rates vs Maturity', fontsize=13)
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
save_figure(fig, "sweep_maturity")
plt.show()

---

## 3. Combined Parameter Exploration

Now let's do a grid search over the most impactful parameters.

In [ ]:
# 2D Grid search: dimension vs volatility
print("=" * 70)
print("GRID SEARCH: Dimension × Volatility")
print("=" * 70)

grid_dims = [5, 10, 15]
grid_sigmas = [0.20, 0.30, 0.40]

grid_results = np.zeros((len(grid_dims), len(grid_sigmas)))
grid_alpha = np.zeros((len(grid_dims), len(grid_sigmas)))
grid_beta = np.zeros((len(grid_dims), len(grid_sigmas)))

for i, d in enumerate(grid_dims):
    for j, sigma in enumerate(grid_sigmas):
        params = create_custom_params(
            d=d, base_sigma=sigma, rho=0.75, T=1.0, max_degree=3
        )
        results = run_mlmc_diagnostics(params, verbose=False)
        
        grid_results[i, j] = results['level_fractions'][0] * 100
        grid_alpha[i, j] = results['diagnostics']['alpha']
        grid_beta[i, j] = results['diagnostics']['beta']
        
        print(f"d={d}, σ={sigma:.2f}: L0={grid_results[i,j]:.1f}%, α={grid_alpha[i,j]:.2f}, β={grid_beta[i,j]:.2f}")

In [ ]:
# Visualise as heatmap
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Level 0 fraction heatmap
ax1 = axes[0]
im1 = ax1.imshow(grid_results, cmap='RdYlGn_r', aspect='auto', 
                  vmin=50, vmax=100, origin='lower')
ax1.set_xticks(range(len(grid_sigmas)))
ax1.set_xticklabels([f'{s:.2f}' for s in grid_sigmas])
ax1.set_yticks(range(len(grid_dims)))
ax1.set_yticklabels([str(d) for d in grid_dims])
ax1.set_xlabel('Base Volatility σ', fontsize=12)
ax1.set_ylabel('Dimension d', fontsize=12)
ax1.set_title('Level 0 Contribution (%)', fontsize=13)
for i in range(len(grid_dims)):
    for j in range(len(grid_sigmas)):
        ax1.text(j, i, f'{grid_results[i,j]:.0f}%', ha='center', va='center', 
                fontsize=12, fontweight='bold')
plt.colorbar(im1, ax=ax1)

# Alpha heatmap
ax2 = axes[1]
im2 = ax2.imshow(grid_alpha, cmap='coolwarm', aspect='auto', 
                  vmin=-0.5, vmax=1.5, origin='lower')
ax2.set_xticks(range(len(grid_sigmas)))
ax2.set_xticklabels([f'{s:.2f}' for s in grid_sigmas])
ax2.set_yticks(range(len(grid_dims)))
ax2.set_yticklabels([str(d) for d in grid_dims])
ax2.set_xlabel('Base Volatility σ', fontsize=12)
ax2.set_ylabel('Dimension d', fontsize=12)
ax2.set_title(r'Weak Convergence Rate $\alpha$', fontsize=13)
for i in range(len(grid_dims)):
    for j in range(len(grid_sigmas)):
        ax2.text(j, i, f'{grid_alpha[i,j]:.2f}', ha='center', va='center', 
                fontsize=12, fontweight='bold')
plt.colorbar(im2, ax=ax2)

# Beta heatmap
ax3 = axes[2]
im3 = ax3.imshow(grid_beta, cmap='coolwarm', aspect='auto', 
                  vmin=0.5, vmax=3.0, origin='lower')
ax3.set_xticks(range(len(grid_sigmas)))
ax3.set_xticklabels([f'{s:.2f}' for s in grid_sigmas])
ax3.set_yticks(range(len(grid_dims)))
ax3.set_yticklabels([str(d) for d in grid_dims])
ax3.set_xlabel('Base Volatility σ', fontsize=12)
ax3.set_ylabel('Dimension d', fontsize=12)
ax3.set_title(r'Variance Decay Rate $\beta$', fontsize=13)
for i in range(len(grid_dims)):
    for j in range(len(grid_sigmas)):
        ax3.text(j, i, f'{grid_beta[i,j]:.2f}', ha='center', va='center', 
                fontsize=12, fontweight='bold')
plt.colorbar(im3, ax=ax3)

plt.tight_layout()
save_figure(fig, "grid_search_dim_sigma")
plt.show()

---

## 4. Testing Candidate Stress Test Configurations

Based on the sweeps, let's test some promising configurations.

In [ ]:
# Define candidate configurations
candidates = {
    'Moderate': {
        'd': 10, 'base_sigma': 0.30, 'rho': 0.75, 'T': 1.0, 'max_degree': 3
    },
    'High-Vol': {
        'd': 10, 'base_sigma': 0.40, 'rho': 0.75, 'T': 1.0, 'max_degree': 3
    },
    'High-Dim': {
        'd': 15, 'base_sigma': 0.30, 'rho': 0.75, 'T': 1.0, 'max_degree': 3
    },
    'High-Corr': {
        'd': 10, 'base_sigma': 0.30, 'rho': 0.90, 'T': 1.0, 'max_degree': 3
    },
    'Long-Maturity': {
        'd': 10, 'base_sigma': 0.30, 'rho': 0.75, 'T': 2.0, 'max_degree': 3
    },
    'High-Degree': {
        'd': 10, 'base_sigma': 0.30, 'rho': 0.75, 'T': 1.5, 'max_degree': 4
    },
    'Aggressive': {
        'd': 15, 'base_sigma': 0.40, 'rho': 0.85, 'T': 1.5, 'max_degree': 4
    },
}

print("=" * 70)
print("CANDIDATE STRESS TEST CONFIGURATIONS")
print("=" * 70)

candidate_results = {}

for name, config in candidates.items():
    print(f"\n{'-'*50}")
    print(f"{name}: d={config['d']}, σ={config['base_sigma']}, ρ={config['rho']}, T={config['T']}, deg={config['max_degree']}")
    print(f"{'-'*50}")
    
    params = create_custom_params(**config)
    results = run_mlmc_diagnostics(params, verbose=True)
    
    candidate_results[name] = {
        'config': config,
        'params': params,
        'results': results
    }
    
    print(f"\n  Level fractions: {[f'{f*100:.1f}%' for f in results['level_fractions']]}")
    print(f"  α = {results['diagnostics']['alpha']:.3f}")
    print(f"  β = {results['diagnostics']['beta']:.3f}")
    print(f"  Quality: {results['diagnostics']['convergence_quality']}")

In [ ]:
# Summary comparison table
print("\n" + "=" * 90)
print("CANDIDATE COMPARISON SUMMARY")
print("=" * 90)
print(f"{'Name':<15} {'L0%':>6} {'L1%':>6} {'L2%':>6} {'L3%':>6} {'α':>8} {'β':>8} {'Quality':<12}")
print("-" * 90)

for name, data in candidate_results.items():
    fracs = data['results']['level_fractions']
    diag = data['results']['diagnostics']
    
    # Pad fractions to 4 levels
    while len(fracs) < 4:
        fracs.append(0.0)
    
    print(f"{name:<15} {fracs[0]*100:>5.1f}% {fracs[1]*100:>5.1f}% {fracs[2]*100:>5.1f}% {fracs[3]*100:>5.1f}% "
          f"{diag['alpha']:>8.3f} {diag['beta']:>8.3f} {diag['convergence_quality']:<12}")

print("=" * 90)
print("\nTarget: L0~60%, L1~25%, L2~10%, L3~5%, α≈1.0, β≈2.0")

In [ ]:
# Plot diagnostics for the most promising candidates
best_candidates = ['Moderate', 'High-Vol', 'Aggressive']

for name in best_candidates:
    data = candidate_results[name]
    config = data['config']
    
    title = f"\n({name}: d={config['d']}, σ={config['base_sigma']}, ρ={config['rho']}, T={config['T']})"
    
    fig = plot_level_diagnostics(data['results'], data['params'], title_suffix=title)
    save_figure(fig, f"candidate_{name.lower().replace('-', '_')}")
    plt.show()

---

## 5. Final Recommendation

Based on the exploration, select the best configuration.

In [ ]:
# Score candidates based on how close they are to ideal
# Ideal: L0≈60%, α≈1.0, β≈2.0, quality='good'

def score_candidate(data):
    """Score a candidate config. Lower is better."""
    fracs = data['results']['level_fractions']
    diag = data['results']['diagnostics']
    
    # Penalties
    l0_penalty = abs(fracs[0] - 0.60) * 10  # Want L0 ~ 60%
    alpha_penalty = abs(diag['alpha'] - 1.0) * 5 if not np.isnan(diag['alpha']) else 10
    beta_penalty = abs(diag['beta'] - 2.0) * 3 if not np.isnan(diag['beta']) else 10
    
    quality_penalty = {'good': 0, 'acceptable': 2, 'poor': 5}.get(diag['convergence_quality'], 10)
    
    return l0_penalty + alpha_penalty + beta_penalty + quality_penalty

scores = {name: score_candidate(data) for name, data in candidate_results.items()}
best_name = min(scores, key=scores.get)

print("=" * 70)
print("CANDIDATE SCORES (lower is better)")
print("=" * 70)
for name, score in sorted(scores.items(), key=lambda x: x[1]):
    marker = " ← BEST" if name == best_name else ""
    print(f"  {name:<15}: {score:.2f}{marker}")

print(f"\n{'='*70}")
print(f"RECOMMENDED STRESS TEST CONFIGURATION: {best_name}")
print(f"{'='*70}")
print(f"\nConfiguration:")
for k, v in candidate_results[best_name]['config'].items():
    print(f"  {k}: {v}")

In [ ]:
# Generate code snippet for config.py
best_config = candidate_results[best_name]['config']

code_snippet = f'''
# =============================================================================
# ADD THIS TO config.py
# =============================================================================

def create_stress_test_params(
    d: int = {best_config['d']},
    base_sigma: float = {best_config['base_sigma']},
    rho: float = {best_config['rho']},
    T: float = {best_config['T']},
    max_degree: int = {best_config['max_degree']},
    seed: int = 42
) -> ProblemParameters:
    """
    Parameters designed to stress-test MLMC convergence.
    
    Creates a complex volatility surface requiring meaningful
    contributions from finer MLMC levels.
    
    Based on stress_test_exploration.ipynb results.
    """
    np.random.seed(seed)
    
    # Varying volatilities
    sigma_multipliers = 0.7 + 0.6 * np.random.rand(d)
    sigma = base_sigma * sigma_multipliers
    
    # Toeplitz correlation matrix
    corr_matrix = np.zeros((d, d))
    for i in range(d):
        for j in range(d):
            corr_matrix[i, j] = rho ** abs(i - j)
    
    # Safe h0: h0 ≤ T / (2*max_degree + 5)
    h0 = 0.8 * T / (2 * max_degree + 5)
    
    return ProblemParameters(
        r=0.05,
        sigma=sigma,
        corr_matrix=corr_matrix,
        P1=np.ones(d),
        x0=100.0 * np.ones(d),
        T=T,
        K=100.0 * d,
        max_degree=max_degree,
        h0=h0,
        random_seed=seed
    )


STRESS_TEST_PARAMS = create_stress_test_params()
'''

print(code_snippet)

---

## 6. Final Comparison: Baseline vs Stress Test

In [ ]:
# Side-by-side comparison
fig, axes = plt.subplots(2, 4, figsize=(20, 10))

# Baseline (top row)
baseline = baseline_results
stress = candidate_results[best_name]['results']

datasets = [
    ('Baseline (Eq. 56)', baseline, DEFAULT_PARAMS),
    (f'Stress Test ({best_name})', stress, candidate_results[best_name]['params'])
]

for row, (title, results, params) in enumerate(datasets):
    n_levels = params.max_degree + 1
    levels = list(range(n_levels))
    corrections = results['corrections']
    fractions = results['level_fractions']
    variances = [results['level_stats'][l].variance for l in levels]
    correlations = [results['level_stats'][l].correlation for l in levels]
    alpha = results['diagnostics']['alpha']
    beta = results['diagnostics']['beta']
    
    # Corrections bar chart
    ax1 = axes[row, 0]
    bars = ax1.bar(levels, corrections, color='steelblue', edgecolor='black', alpha=0.7)
    ax1.set_xlabel(r'Level $\ell$')
    ax1.set_ylabel(r'Mean correction $m_\ell$')
    ax1.set_title(f'{title}\nLevel Corrections')
    ax1.set_xticks(levels)
    ax1.grid(True, alpha=0.3, axis='y')
    
    # Fraction pie chart
    ax2 = axes[row, 1]
    colors = plt.cm.Blues(np.linspace(0.3, 0.9, n_levels))
    wedges, texts, autotexts = ax2.pie(
        fractions, labels=[f'L{l}' for l in levels],
        autopct='%1.1f%%', colors=colors, startangle=90
    )
    ax2.set_title('Level Contributions')
    
    # Weak convergence
    ax3 = axes[row, 2]
    log2_m = [np.log2(abs(c)) if abs(c) > 1e-15 else np.nan for c in corrections]
    ax3.plot(levels, log2_m, 'bo-', markersize=8, linewidth=2)
    ax3.set_xlabel(r'Level $\ell$')
    ax3.set_ylabel(r'$\log_2 |m_\ell|$')
    ax3.set_title(f'Weak Convergence (α = {alpha:.2f})')
    ax3.set_xticks(levels)
    ax3.grid(True, alpha=0.3)
    
    # Variance decay
    ax4 = axes[row, 3]
    higher_levels = [l for l in levels if l >= 1]
    log2_v = [np.log2(variances[l]) if variances[l] > 1e-15 else np.nan for l in higher_levels]
    ax4.plot(higher_levels, log2_v, 'go-', markersize=8, linewidth=2)
    ax4.set_xlabel(r'Level $\ell$')
    ax4.set_ylabel(r'$\log_2 V_\ell$')
    ax4.set_title(f'Variance Decay (β = {beta:.2f})')
    if len(higher_levels) > 0:
        ax4.set_xticks(higher_levels)
    ax4.grid(True, alpha=0.3)

plt.tight_layout()
save_figure(fig, "final_comparison")
plt.show()

In [ ]:
# Final summary
print("\n" + "=" * 70)
print("STRESS TEST EXPLORATION COMPLETE")
print("=" * 70)
print(f"\nRecommended configuration: {best_name}")
print(f"\nBaseline (Eq. 56):")
print(f"  Level 0: {baseline_results['level_fractions'][0]*100:.1f}%")
print(f"  α = {baseline_results['diagnostics']['alpha']:.3f}")
print(f"  β = {baseline_results['diagnostics']['beta']:.3f}")

print(f"\nStress Test ({best_name}):")
print(f"  Level 0: {stress['level_fractions'][0]*100:.1f}%")
print(f"  α = {stress['diagnostics']['alpha']:.3f}")
print(f"  β = {stress['diagnostics']['beta']:.3f}")

print(f"\nNext steps:")
print(f"  1. Add create_stress_test_params() to config.py")
print(f"  2. Run: python run_all_experiments.py --exp 2 (with stress params)")
print(f"  3. Compare results with baseline")
print("=" * 70)